# TTA Probability Probes

Notebook này chạy từng method và lưu per-sample probability để soi sâu AUC/ranking. Output chính là long-table: mỗi dòng = một sample dưới một `dataset/model/method/param_id`.

## Kaggle Setup

In [ ]:
# Chạy cell này trên Kaggle nếu repo chưa có trong /kaggle/working.
# !git clone -b dev https://github.com/hoavien0110/training-free-tta-for-deepfake-detection.git /kaggle/working/training-free-tta-for-deepfake-detection
# %cd /kaggle/working/training-free-tta-for-deepfake-detection
# !pip install -q -e . --no-deps


## Imports

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json
import shlex
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, roc_auc_score

repo_root = Path.cwd()
sys.path.insert(0, str(repo_root))

from deepfake_tta.modeling import load_feature_file, predict_probe_scores, seed_everything
from testing.evaluate_tta_matrix import (
    apply_aligned_ids,
    build_aligned_balanced_ids,
    build_sample_index,
    find_feature_files,
    infer_feature_meta,
    load_probe_model,
    parse_spec,
    read_thresholds,
)
from testing.evaluate_tta_param_sweep import create_method


## Config

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
EVAL_BATCH_SIZE = 4096
TEST_BATCH_SIZE = 512
CACHE_BATCH_SIZE = 8192
BLOCK_SIZE = 16
BALANCE_METHOD_FIT = True

# Same input layout as the sweep notebooks. Trên local thì đổi các path này sang file/folder có sẵn.
FFPP_SPLIT = "/kaggle/input/datasets/vhonghoavin/ffpp-split-features"
FFPP_CORR = "/kaggle/input/datasets/vhonghoavin/ffpp-test-embeddings/ffpp_test_embeddings"
CELEB_CORR = "/kaggle/input/datasets/vhonghoavin/deepfakebench-features"
MODELS_DIR = "/kaggle/input/datasets/vhonghoavin/ffpp-training-free-models"
MODELS_BAL = "/kaggle/input/datasets/vhonghoavin/ffpp-training-free-models-balanced"

TRAIN_FEATURES = f"{FFPP_SPLIT}/ffpp_train_features.pt"
DATASETS = [
    f"ffpp-test={FFPP_SPLIT}/ffpp_test_features.pt",
    f"ffpp-test-corruption={FFPP_CORR}",
    f"celebdfv1-test-corruption={CELEB_CORR}",
    f"ffpp-test-balanced={FFPP_SPLIT}/ffpp_test_features.pt",
    f"ffpp-test-corruption-balanced={FFPP_CORR}",
    f"celebdfv1-test-corruption-balanced={CELEB_CORR}",
]
MODELS = [
    f"linear-probe={MODELS_DIR}/ffpp_linear_probe_split.pt",
    f"osd={MODELS_DIR}/ffpp_osd_linear_probe_split.pt",
    f"linear-probe-balanced={MODELS_BAL}/ffpp_linear_probe_split.pt",
    f"osd-balanced={MODELS_BAL}/ffpp_osd_linear_probe_split.pt",
]
THRESHOLDS_CSV = [
    f"linear-probe={MODELS_DIR}/thresholds.csv",
    f"osd={MODELS_DIR}/thresholds.csv",
    f"linear-probe-balanced={MODELS_BAL}/thresholds.csv",
    f"osd-balanced={MODELS_BAL}/thresholds.csv",
]

# Set dataset name ở đây nếu muốn notebook balance/aligned giống các sweep cũ.
BALANCED_DATASETS = {"ffpp-test-balanced"}
BALANCED_ALIGNED_DATASETS = {
    "ffpp-test-corruption-balanced",
    "celebdfv1-test-corruption-balanced",
}
SHUFFLE_DATASETS = set()
SHUFFLE_ALL_TESTS = False

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("eda/figures/tta_probability_probes")
SPLIT_PROBE_DIR = OUTPUT_DIR / "tta_probability_probe_files"
PROBES_CSV = OUTPUT_DIR / "tta_probability_probes.csv"
SUMMARY_CSV = OUTPUT_DIR / "tta_probability_probe_summary.csv"

seed_everything(SEED)
print("device:", DEVICE)


## Methods To Probe

In [ ]:
METHOD_CONFIGS = [
    {"method": "none", "param_id": "none"},
    {"method": "dota", "param_id": "dota_bw0.55_m0.95_ct0.9", "base_weight": 0.55, "momentum": 0.95, "confidence_threshold": 0.9},
]

TTA_ARGS = SimpleNamespace(
    seed=SEED,
    eval_batch_size=EVAL_BATCH_SIZE,
    test_batch_size=TEST_BATCH_SIZE,
    cache_batch_size=CACHE_BATCH_SIZE,
    freetta_min_var=1e-4,
    freetta_warmup_batches=1,
    bca_confidence_threshold=0.0,
    dota_min_var=1e-4,
    tda_positive_shot_capacity=64,
    tda_negative_beta=5.5,
    tda_negative_shot_capacity=64,
    tda_negative_entropy_lower=0.35,
    tda_negative_entropy_upper=0.8,
    tda_negative_mask_lower=0.2,
    tda_negative_mask_upper=0.8,
    tda_top_k=64,
    gda_min_var=1e-4,
    etta_entropy_power=1.0,
    etta_base_weight=1.0,
)

pd.DataFrame(METHOD_CONFIGS)


## Helpers

In [ ]:
def sample_ids_from_payload(payload, n):
    paths = payload.get("paths")
    if paths is None:
        return [f"idx_{idx:06d}" for idx in range(n)]
    sample_index = build_sample_index(payload)
    by_position = {idx: sample_id for sample_id, idx in sample_index.items()}
    return [by_position.get(idx, str(paths[idx])) for idx in range(n)]


def select_balanced_subset_with_ids(feats, labels, sample_ids, *, seed, name):
    labels = labels.long()
    counts = torch.bincount(labels, minlength=2)
    keep_per_class = int(counts.min().item())
    if keep_per_class <= 0:
        raise ValueError(f"Cannot balance {name} with label counts: {counts.tolist()}")

    generator = torch.Generator()
    generator.manual_seed(seed)
    selected = []
    for cls_idx in range(len(counts)):
        cls_indices = torch.where(labels == cls_idx)[0]
        perm = torch.randperm(len(cls_indices), generator=generator)
        selected.append(cls_indices[perm[:keep_per_class]])
    indices = torch.cat(selected)
    indices = indices[torch.randperm(len(indices), generator=generator)]
    ids = [sample_ids[int(idx)] for idx in indices]
    return feats[indices].contiguous(), labels[indices].contiguous(), ids


def shuffle_with_ids(feats, labels, sample_ids, *, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    indices = torch.randperm(len(labels), generator=generator)
    ids = [sample_ids[int(idx)] for idx in indices]
    return feats[indices].contiguous(), labels[indices].contiguous(), ids


def metric_or_nan(fn, *args):
    try:
        return float(fn(*args))
    except ValueError:
        return np.nan


def summarize_scores(y_true, y_score, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "auc": metric_or_nan(roc_auc_score, y_true, y_score),
        "ap": metric_or_nan(average_precision_score, y_true, y_score),
    }


def threshold_for_model(thresholds, model_name, model_type, model_path):
    return thresholds.get(
        model_name,
        thresholds.get(
            f"{model_name}:{model_type}",
            thresholds.get(f"{model_name}:{model_path.name}", thresholds.get(model_path.name, thresholds.get(model_type, 0.5))),
        ),
    )


def probability_rows(common, sample_ids, labels, probs, y_pred, threshold, fit_samples, fit_counts):
    y_true = labels.detach().cpu().numpy().astype(int)
    probs = probs.detach().cpu().numpy()
    rows = []
    for idx, sample_id in enumerate(sample_ids):
        prob_real = float(probs[idx, 0])
        prob_fake = float(probs[idx, 1])
        rows.append({
            **common,
            "sample_index": idx,
            "sample_id": sample_id,
            "y_true": int(y_true[idx]),
            "prob_real": prob_real,
            "prob_fake": prob_fake,
            "score": prob_fake,
            "y_pred": int(y_pred[idx]),
            "correct": bool(int(y_pred[idx]) == int(y_true[idx])),
            "confidence": float(max(prob_real, prob_fake)),
            "margin": float(abs(prob_fake - prob_real)),
            "threshold": float(threshold),
            "method_fit_samples": fit_samples,
            "method_fit_counts": fit_counts,
        })
    return rows


## Run And Save Probes

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_PROBE_DIR.mkdir(parents=True, exist_ok=True)

METHOD_CONFIG_ARGS = " ".join(
    f"--method-config-json {shlex.quote(json.dumps(config, separators=(',', ':')))}"
    for config in METHOD_CONFIGS
)
DATASET_ARGS = " ".join(f"--dataset {shlex.quote(dataset)}" for dataset in DATASETS)
MODEL_ARGS = " ".join(f"--model {shlex.quote(model)}" for model in MODELS)
THRESHOLD_ARGS = " ".join(f"--thresholds-csv {shlex.quote(path)}" for path in THRESHOLDS_CSV)
BALANCED_DATASET_ARGS = " ".join(f"--balanced-dataset {shlex.quote(name)}" for name in sorted(BALANCED_DATASETS))
BALANCED_ALIGNED_ARGS = " ".join(f"--balanced-aligned-dataset {shlex.quote(name)}" for name in sorted(BALANCED_ALIGNED_DATASETS))
SHUFFLE_DATASET_ARGS = " ".join(f"--shuffle-dataset {shlex.quote(name)}" for name in sorted(SHUFFLE_DATASETS))
SHUFFLE_ALL_TESTS_ARG = "--shuffle-all-tests" if SHUFFLE_ALL_TESTS else ""
BALANCE_METHOD_FIT_ARG = "--balance-method-fit" if BALANCE_METHOD_FIT else ""
SCRIPT_PATH = "testing/evaluate_tta_probability_probes.py"
if not Path(SCRIPT_PATH).exists():
    raise FileNotFoundError("Cannot find evaluate_tta_probability_probes.py. Check the notebook working directory.")

print("Method config args:")
print(METHOD_CONFIG_ARGS)

!python -u {SCRIPT_PATH} \
  --train-features {TRAIN_FEATURES} \
  {DATASET_ARGS} \
  {MODEL_ARGS} \
  {THRESHOLD_ARGS} \
  {BALANCED_DATASET_ARGS} \
  {BALANCED_ALIGNED_ARGS} \
  {SHUFFLE_DATASET_ARGS} \
  {SHUFFLE_ALL_TESTS_ARG} \
  {BALANCE_METHOD_FIT_ARG} \
  {METHOD_CONFIG_ARGS} \
  --eval-batch-size {EVAL_BATCH_SIZE} \
  --test-batch-size {TEST_BATCH_SIZE} \
  --cache-batch-size {CACHE_BATCH_SIZE} \
  --block-size {BLOCK_SIZE} \
  --device {DEVICE} \
  --seed {SEED} \
  --continue-on-error \
  --split-probe-dir {SPLIT_PROBE_DIR} \
  --probes-output {PROBES_CSV} \
  --summary-output {SUMMARY_CSV}

split_files = sorted(SPLIT_PROBE_DIR.glob("*.csv"))
if not split_files or not SUMMARY_CSV.exists():
    raise FileNotFoundError(
        "CLI did not create output CSVs. Check the !python log above; the real error is usually above this cell. "
        f"split_files={len(split_files)} summary={SUMMARY_CSV.exists()}"
    )

probes = pd.concat([pd.read_csv(path) for path in split_files], ignore_index=True)
summary = pd.read_csv(SUMMARY_CSV)
print("saved split probe files:", len(split_files), "under", SPLIT_PROBE_DIR)
print("combined preview shape:", probes.shape)
print("saved summary:", SUMMARY_CSV, summary.shape)
display(pd.DataFrame({"file": [str(path) for path in split_files]}).head(30))
display(summary.sort_values(["dataset", "model", "auc"], ascending=[True, True, False]).head(40))
display(probes.head())


## Quick AUC Deep Dive

In [ ]:
split_files = sorted(SPLIT_PROBE_DIR.glob("*.csv"))
probes = pd.concat([pd.read_csv(path) for path in split_files], ignore_index=True)

group_cols = ["dataset", "corruption", "level", "model", "method", "param_id"]
auc_table = (
    probes.groupby(group_cols, dropna=False)
    .apply(lambda g: roc_auc_score(g["y_true"], g["score"]) if g["y_true"].nunique() == 2 else np.nan)
    .reset_index(name="auc")
    .sort_values("auc", ascending=False)
)
display(auc_table.head(30))

# Các case làm đau AUC: fake score thấp hoặc real score cao.
hard_positive = probes[probes["y_true"] == 1].sort_values("score", ascending=True).head(30)
hard_negative = probes[probes["y_true"] == 0].sort_values("score", ascending=False).head(30)
display(hard_positive)
display(hard_negative)


## Same-Sample Method Comparison

In [ ]:
key_cols = ["dataset", "feature_path", "model", "sample_id", "y_true"]
wide = probes.pivot_table(
    index=key_cols,
    columns="param_id",
    values="score",
    aggfunc="first",
).reset_index()

score_cols = [col for col in wide.columns if col not in key_cols]
wide["score_range_across_methods"] = wide[score_cols].max(axis=1) - wide[score_cols].min(axis=1)
display(wide.sort_values("score_range_across_methods", ascending=False).head(40))
